# Kaggle Heart Disease Prediction - The Winning Solution

## Goal
Surpass the top leaderboard score (0.95405).

## Strategy
1. **Pre-Training**: Train XGBoost, LightGBM, CatBoost with **Optuna** tuned parameters.
2. **Pseudo-Labeling**: Use the ensemble's high-confidence predictions on the Test set to augment the Training set.
3. **Retraining**: Train the models again on the larger (Train + Pseudo) dataset.
4. **Final Ensemble**: Weighted average of the retrained models.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
import warnings

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler, PolynomialFeatures
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize
from copy import deepcopy

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option('display.max_columns', None)

# Constants
SEED = 42
N_FOLDS = 10
TARGET = 'Heart Disease'
N_TRIALS = 30  # Keep reasonable for finding good params
 PSEUDO_THRESHOLD_UPPER = 0.99 # Confidence threshold for Positive (1)
PSEUDO_THRESHOLD_LOWER = 0.01 # Confidence threshold for Negative (0)

## 1. Data Loading & Advanced Feature Engineering

In [ ]:
train_full = pd.read_csv('train.csv')
test_full = pd.read_csv('test.csv')
train_full[TARGET] = train_full[TARGET].map({'Presence': 1, 'Absence': 0})

def advanced_feature_engineering(df):
    df = df.copy()
    # --- Domain Knowledge Interactions ---
    df['BP_Age_Ratio'] = df['BP'] / (df['Age'] + 1)
    df['BP_Cholesterol_Product'] = df['BP'] * df['Cholesterol']
    df['MaxHR_Age_Ratio'] = df['Max HR'] / (df['Age'] + 1)
    df['HeartRate_Reserve'] = 220 - df['Age'] - df['Max HR'] 
    df['Cholesterol_Age_Ratio'] = df['Cholesterol'] / (df['Age'] + 1)
    df['ST_Depression_Slope'] = df['ST depression'] * df['Slope of ST']
    
    # --- Log Transforms ---
    for col in ['Cholesterol', 'ST depression']:
        df[f'Log_{col}'] = np.log1p(df[col])
        
    # --- Polynomial Features ---
    poly_cols = ['Age', 'Max HR', 'Cholesterol', 'BP', 'ST depression']
    poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
    poly_features = poly.fit_transform(df[poly_cols])
    poly_feature_names = poly.get_feature_names_out(poly_cols)
    
    poly_df = pd.DataFrame(poly_features, columns=[f'Poly_{col}' for col in poly_feature_names])
    df = df.reset_index(drop=True)
    poly_df = poly_df.reset_index(drop=True)
    df = pd.concat([df, poly_df], axis=1)
    return df

train_eng = advanced_feature_engineering(train_full)
test_eng = advanced_feature_engineering(test_full)

cols_to_use = [c for c in train_eng.columns if c not in ['id', TARGET]]
X = train_eng[cols_to_use]
y = train_eng[TARGET]
X_test = test_eng[cols_to_use]

print(f"Modeling with {len(cols_to_use)} features.")

## 2. Optuna Hyperparameter Tuning (Initial Phase)

In [ ]:
# Define Objective Functions (Manual CV for Stability)
def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 2000),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'random_state': SEED, 'n_jobs': -1, 'eval_metric': 'auc', 'early_stopping_rounds': 50
    }
    scores = []
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    for tr_i, val_i in skf.split(X, y):
        model = XGBClassifier(**params)
        model.fit(X.iloc[tr_i], y.iloc[tr_i], eval_set=[(X.iloc[val_i], y.iloc[val_i])], verbose=False)
        scores.append(roc_auc_score(y.iloc[val_i], model.predict_proba(X.iloc[val_i])[:, 1]))
    return np.mean(scores)

def objective_lgbm(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 2000),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'max_depth': trial.suggest_int('max_depth', -1, 15),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'random_state': SEED, 'n_jobs': -1, 'metric': 'auc', 'verbosity': -1
    }
    scores = []
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    for tr_i, val_i in skf.split(X, y):
        model = LGBMClassifier(**params)
        model.fit(X.iloc[tr_i], y.iloc[tr_i], eval_set=[(X.iloc[val_i], y.iloc[val_i])], callbacks=[])
        scores.append(roc_auc_score(y.iloc[val_i], model.predict_proba(X.iloc[val_i])[:, 1]))
    return np.mean(scores)

def objective_cat(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 500, 2000),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        'random_strength': trial.suggest_float('random_strength', 1e-9, 10),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0, 1),
        'random_seed': SEED, 'eval_metric': 'AUC', 'verbose': 0, 'allow_writing_files': False, 'early_stopping_rounds': 50
    }
    scores = []
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    for tr_i, val_i in skf.split(X, y):
        model = CatBoostClassifier(**params)
        model.fit(X.iloc[tr_i], y.iloc[tr_i], eval_set=(X.iloc[val_i], y.iloc[val_i]), verbose=False)
        scores.append(roc_auc_score(y.iloc[val_i], model.predict_proba(X.iloc[val_i])[:, 1]))
    return np.mean(scores)

print("Running Optuna (Initial)...")
study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS)

study_lgbm = optuna.create_study(direction='maximize')
study_lgbm.optimize(objective_lgbm, n_trials=N_TRIALS)

study_cat = optuna.create_study(direction='maximize')
study_cat.optimize(objective_cat, n_trials=N_TRIALS)

best_xgb = study_xgb.best_params
best_lgbm = study_lgbm.best_params
best_cat = study_cat.best_params

# Add static params back
best_xgb.update({'random_state': SEED, 'n_jobs': -1, 'eval_metric': 'auc', 'early_stopping_rounds': 100})
best_lgbm.update({'random_state': SEED, 'n_jobs': -1, 'metric': 'auc', 'verbosity': -1})
best_cat.update({'random_seed': SEED, 'eval_metric': 'AUC', 'verbose': 0, 'allow_writing_files': False, 'early_stopping_rounds': 100})

## 3. Initial Ensemble & Predictions (For Pseudo-Labeling)

In [ ]:
def train_predict(X, y, X_test, params, model_type):
    oof_preds = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    
    for fold, (tr_i, val_i) in enumerate(skf.split(X, y), 1):
        X_tr, y_tr = X.iloc[tr_i], y.iloc[tr_i]
        X_va, y_va = X.iloc[val_i], y.iloc[val_i]
        
        if model_type == 'xgb':
            model = XGBClassifier(**params)
            model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        elif model_type == 'lgbm':
            model = LGBMClassifier(**params)
            model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[])
        elif model_type == 'cat':
            model = CatBoostClassifier(**params)
            model.fit(X_tr, y_tr, eval_set=(X_va, y_va), verbose=False)
            
        val_p = model.predict_proba(X_va)[:, 1]
        test_p = model.predict_proba(X_test)[:, 1]
        
        oof_preds[val_i] = val_p
        test_preds += test_p / N_FOLDS
        
    return oof_preds, test_preds

print("Training Initial Models...")
xgb_oof, xgb_pred = train_predict(X, y, X_test, best_xgb, 'xgb')
lgbm_oof, lgbm_pred = train_predict(X, y, X_test, best_lgbm, 'lgbm')
cat_oof, cat_pred = train_predict(X, y, X_test, best_cat, 'cat')

# Initial Ensemble Weights
oof_stack = np.column_stack([xgb_oof, lgbm_oof, cat_oof])
test_stack = np.column_stack([xgb_pred, lgbm_pred, cat_pred])

def loss_func(w):
    w = w / np.sum(w)
    return -roc_auc_score(y, np.dot(oof_stack, w))

res = minimize(loss_func, [1/3]*3, method='Nelder-Mead')
best_w = res.x / np.sum(res.x)
print(f"Initial Best Weights: {best_w}")

initial_test_preds = np.dot(test_stack, best_w)

## 4. Pseudo-Labeling
Expanding the training set with confident test predictions.

In [ ]:
# Identify confident samples
pseudo_mask_pos = initial_test_preds > PSEUDO_THRESHOLD_UPPER
pseudo_mask_neg = initial_test_preds < PSEUDO_THRESHOLD_LOWER

X_pseudo_pos = X_test[pseudo_mask_pos].copy()
y_pseudo_pos = pd.Series(1, index=X_pseudo_pos.index)

X_pseudo_neg = X_test[pseudo_mask_neg].copy()
y_pseudo_neg = pd.Series(0, index=X_pseudo_neg.index)

n_pos = len(X_pseudo_pos)
n_neg = len(X_pseudo_neg)
print(f"Adding {n_pos} Positive and {n_neg} Negative Pseudo-Labels.")

# Combine with original train
X_augmented = pd.concat([X, X_pseudo_pos, X_pseudo_neg], axis=0).reset_index(drop=True)
y_augmented = pd.concat([y, y_pseudo_pos, y_pseudo_neg], axis=0).reset_index(drop=True)

print(f"New Train Shape: {X_augmented.shape}")

## 5. Retraining on Augmented Data

In [ ]:
print("Retraining Models with Pseudo-Labels...")
# We use the same best parameters found earlier
xgb_oof_aug, xgb_pred_final = train_predict(X_augmented, y_augmented, X_test, best_xgb, 'xgb')
lgbm_oof_aug, lgbm_pred_final = train_predict(X_augmented, y_augmented, X_test, best_lgbm, 'lgbm')
cat_oof_aug, cat_pred_final = train_predict(X_augmented, y_augmented, X_test, best_cat, 'cat')

# Ensemble again (Using weights optimized on original data is usually safer, but we can re-optimize on augmented OOF)
# Let's re-optimize on augmented OOF for maximum fit
oof_stack_aug = np.column_stack([xgb_oof_aug, lgbm_oof_aug, cat_oof_aug])
test_stack_final = np.column_stack([xgb_pred_final, lgbm_pred_final, cat_pred_final])

def loss_func_aug(w):
    w = w / np.sum(w)
    return -roc_auc_score(y_augmented, np.dot(oof_stack_aug, w))

res_aug = minimize(loss_func_aug, [1/3]*3, method='Nelder-Mead')
best_w_final = res_aug.x / np.sum(res_aug.x)
print(f"Final Weights: {best_w_final}")

final_predictions = np.dot(test_stack_final, best_w_final)

# Submission
submission['Heart Disease'] = final_predictions
submission.to_csv('submission_final.csv', index=False)
print("submission_final.csv saved! Good luck!")
submission.head()